# Cleaning and preprocessing

Reads the output of `checkpoint1.preprocess` (run it first). Set `CELEBA_SPOOF_ROOT` to the dataset folder.

In [1]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

if Path.cwd().name == 'notebooks':
    os.chdir('..')

import numpy as np
import torch
import torchvision
from sklearn.decomposition import PCA
from torchvision import transforms

from checkpoint1 import data

root = data.check_data_root()
manifest_dir = Path(os.environ.get('MANIFEST_DIR', 'data/processed'))
seed = data.SEED

manifests = {split: pd.read_csv(manifest_dir / f'manifest_{split}.csv') for split in ['train', 'val', 'test']}
dropped = pd.read_csv(manifest_dir / 'dropped_samples.csv')
train = manifests['train']

## Dropped and flagged images

In [2]:
print(f"dropped {len(dropped):,} images")
display(dropped['reason'].value_counts())

all_clean = pd.concat(manifests.values())
print(f"clean images: {len(all_clean):,}")
print(f"face box partly outside the image: {all_clean['flag_partial_bbox'].sum():,}")
print(f"face smaller than 40 px: {all_clean['flag_small_face'].sum():,}")

dropped 4,541 images


reason
missing_image     2518
label_conflict    2022
bad_bbox             1
Name: count, dtype: int64

clean images: 557,034
face box partly outside the image: 152,100
face smaller than 40 px: 654


## Subject-disjoint splits

In [3]:
rows = []
for split, manifest in manifests.items():
    n_spoof = manifest['is_spoof'].sum()
    rows.append({'split': split, 'subjects': manifest['subject'].nunique(), 'images': len(manifest),
                 'live': len(manifest) - n_spoof, 'spoof': n_spoof})
pd.DataFrame(rows)

,split,subjects,images,live,spoof
0,train,7373,442001,145761,296240
1,val,819,50381,16701,33680
2,test,969,64652,19137,45515


## Normalization

Mean and std of face crops on 2,000 training images (half live, half spoof), compared with ImageNet.

In [4]:
def balanced_sample(manifest, n, seed):
    parts = []
    for _, group in manifest.groupby('is_spoof'):
        parts.append(group.sample(n=n // 2, random_state=seed))
    return pd.concat(parts).sort_values('path')


norm_sample = balanced_sample(train, 2000, seed)
pixel_sum = np.zeros(3)
pixel_sq_sum = np.zeros(3)
n_pixels = 0
for _, row in norm_sample.iterrows():
    pixels = np.asarray(data.load_crop(root, row), dtype=np.float64).reshape(-1, 3) / 255
    pixel_sum += pixels.sum(axis=0)
    pixel_sq_sum += (pixels ** 2).sum(axis=0)
    n_pixels += len(pixels)
mean = pixel_sum / n_pixels
std = np.sqrt(pixel_sq_sum / n_pixels - mean ** 2)

pd.DataFrame({'dataset mean': mean, 'ImageNet mean': [0.485, 0.456, 0.406],
              'dataset std': std, 'ImageNet std': [0.229, 0.224, 0.225]}, index=['R', 'G', 'B']).round(3)

,dataset mean,ImageNet mean,dataset std,ImageNet std
R,0.487,0.485,0.296,0.229
G,0.408,0.456,0.269,0.224
B,0.374,0.406,0.264,0.225


## Feature reduction: correlation

Association of the anti-spoof labels with live/spoof (Cramér's V) and of each face attribute (Pearson r).

In [5]:
def cramers_v(x, y):
    table = pd.crosstab(x, y).to_numpy().astype(float)
    expected = table.sum(axis=1, keepdims=True) * table.sum(axis=0, keepdims=True) / table.sum()
    chi2 = ((table - expected) ** 2 / expected).sum()
    return np.sqrt(chi2 / (table.sum() * (min(table.shape) - 1)))


for column in ['spoof_type', 'illumination', 'environment']:
    print(f"{column}: Cramér's V = {cramers_v(train[column], train['is_spoof']):.2f}")

attribute_r = train[data.CELEBA_ATTRS].corrwith(train['is_spoof'])
attribute_r.sort_values(key=abs, ascending=False).head(5).round(2)

spoof_type: Cramér's V = 1.00
illumination: Cramér's V = 1.00
environment: Cramér's V = 1.00


No_Beard              -0.88
Young                 -0.84
Attractive            -0.64
Mouth_Slightly_Open   -0.62
Smiling               -0.62
dtype: float64

In [6]:
# pairs of face attributes with |r| >= 0.7
attribute_corr = train[data.CELEBA_ATTRS].corr()
pairs = []
for i, first in enumerate(data.CELEBA_ATTRS):
    for second in data.CELEBA_ATTRS[i + 1:]:
        correlation = attribute_corr.loc[first, second]
        if abs(correlation) >= 0.7:
            pairs.append({'attribute 1': first, 'attribute 2': second, 'r': round(correlation, 2)})
pd.DataFrame(pairs)

,attribute 1,attribute 2,r
0,Attractive,Young,0.70
1,Heavy_Makeup,Wearing_Lipstick,0.87
2,High_Cheekbones,Smiling,0.80
3,Mouth_Slightly_Open,Smiling,0.71
4,No_Beard,Young,0.77
